# 1. Ingest SEC 10-K filings

This notebook builds the **Bronze** layer: raw SEC filing documents and identifying metadata, stored in the `financial_db.filings.sec_rag_bronze` Delta table.

## Before running

- Configure Databricks Connect for your workspace and use an identity with permission to create/use the `financial_db` catalog and `filings` schema and write Delta tables.
- Set `SEC_USER_AGENT` in the environment (or project `.env`) to an application name and contact email. SEC requests require an identifying User-Agent.
- The ticker set and lookback limit are configured in the first code cell. This notebook requests recent 10-K filings for those tickers, not every SEC issuer or every filing.

### What the cells do

1. **Configure scope and SEC request headers.** Set the catalog, schema, target tickers, request delay, and maximum filings per ticker.

In [ ]:
import requests
import time
from databricks.connect import DatabricksSession
import os
import re
import os
from dotenv import load_dotenv

load_dotenv()

catalog_name = "financial_db"
schema_name = "filings"

sec_user_agent = os.getenv("SEC_USER_AGENT")
if not sec_user_agent:
    raise RuntimeError("Set SEC_USER_AGENT in .env or the notebook environment before making SEC requests.")

HEADERS = {"User-Agent": sec_user_agent}
TICKERS = {"AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", "NVDA"}
TABLE = f"{catalog_name}.{schema_name}.sec_rag_bronze"
RATE_LIMIT_DELAY = 0.12 # ~8 requests per second
YEARS_LIMIT = 5 # Limit to the last 5 years of filings

2. **Connect and prepare storage.** Start a Databricks Connect session and create the catalog, schema, and append-only Bronze table if they do not exist.

In [ ]:
os.environ.pop("DATABRICKS_METADATA_SERVICE_URL", None)
os.environ["DATABRICKS_AUTH_TYPE"] = "pat"
os.environ["DATABRICKS_SERVERLESS_COMPUTE_ID"] = "auto"

spark = DatabricksSession.builder.getOrCreate()

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
spark.sql(f'''
CREATE TABLE IF NOT EXISTS {TABLE} (
    ticker STRING,
    cik STRING,
    date STRING,
    accession_number STRING,
    url STRING,
    raw STRING
)
USING DELTA
TBLPROPERTIES (
    'delta.appendOnly' = 'true',
    'delta.minReaderVersion' = '1',
    'delta.minWriterVersion' = '2'
);
''')

3. **Fetch filing data.** Resolve ticker-to-CIK mappings, request recent filing metadata, download filing documents, remove embedded base64 images to reduce stored payload size, and append successful downloads to Bronze.

In [ ]:
def get_ciks():
    url = "https://www.sec.gov/files/company_tickers.json"

    time.sleep(RATE_LIMIT_DELAY) # SEC limit
    res = requests.get(url, headers=HEADERS, timeout=10)
    if res.status_code != 200:
        return None
    data = res.json()

    return {v["ticker"]: str(v["cik_str"]).zfill(10) for v in data.values()} # Map of tickers with their formatted CIK


def fetch_10ks(cik: str, limit: int = YEARS_LIMIT):
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"

    time.sleep(RATE_LIMIT_DELAY) # SEC limit
    res = requests.get(url, headers=HEADERS, timeout=10)
    if res.status_code != 200:
        return None

    results = [] # Array for storing the filings data, we'll do the requests later
    counter = 0
    
    filings = res.json()["filings"]["recent"]
    for i, form in enumerate(filings["form"]):
        if form == "10-K":
            accession = filings["accessionNumber"][i]
            doc_name = filings["primaryDocument"][i]
            url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession.replace('-', '')}/{doc_name}"
            results.append({
                "accession_number": accession,
                "date": filings["filingDate"][i],
                "url": url
            })
            counter += 1
            if counter >= limit:
                return results
    return results

def get_bronze():
    ciks = get_ciks()
    existing_df = spark.sql(f"SELECT accession_number FROM {TABLE}")
    existing = set(v["accession_number"] for v in existing_df.collect())
    batch = []

    for ticker in TICKERS:
        cik = ciks.get(ticker)
        if not cik:
            continue

        doc_info = fetch_10ks(cik, 1)
        for filing in doc_info:
            if filing["accession_number"] not in existing:
                time.sleep(RATE_LIMIT_DELAY) # SEC limit
                res = requests.get(filing["url"], headers=HEADERS, timeout=30)
                if res.status_code==200:
                    res_cleaned = re.sub(r'data:image/[^;]+;base64,[A-Za-z0-9+/=]+', '', res.text) # Remove images from the document to reduce size
                    batch.append({
                        "ticker": ticker,
                        "cik": cik,
                        "date": filing["date"],
                        "accession_number": filing["accession_number"],
                        "url": filing["url"],
                        "raw": res_cleaned
                    }) # The data we'll have in our table
                else:
                    print(f"Error fetching document")
    if batch:
        df = spark.createDataFrame(batch)
        df.write.format("delta").mode("append").saveAsTable(TABLE)

get_bronze()

4. **Inspect results.** Display filing metadata and raw-text lengths for a quick ingestion check.

In [ ]:
spark.sql(f'''
SELECT ticker, cik, date, accession_number, url, length(raw) AS charCount FROM {TABLE}
ORDER BY ticker ASC, date DESC
''').toPandas()